In [ ]:
from google.colab import files
import os


print("Please upload your .fasta file")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

with open(file_name, "r") as f:
    lines = f.readlines()
    sequence = "".join(line.strip() for line in lines if not line.startswith(">"))

Genome = sequence
def SkewDiagram(Genome): # generating skewdiagram
    skew = [0]
    for base in Genome:
        current_score = skew[-1]
        if base == "G":
            new_score = current_score + 1
        elif base == "C":
            new_score = current_score - 1
        else:
            new_score = current_score
        skew.append(new_score)
    return skew

def MinimumSkew(Genome): # finding minimum value -> it's location(s)
    result_list = SkewDiagram(Genome)
    min_val = min(result_list) # find minimum value from skewdiagram
    min_indices = [] # new list to put minimum value
    for i in range(len(result_list)): # go through result_list
        if result_list[i] == min_val: # if current value = minimum
            min_indices.append(i) # put into the list
    return min_indices

print("In progress... Please wait!")
print(*(MinimumSkew(Genome)))

min_points = MinimumSkew(Genome)
ori_start = min_points[0]  # we only need the smallest value
target_region = Genome[ori_start - 500 : ori_start + 500] # typically around +/- 500 bp
sequence = target_region # our new sequence to work with!
pl = 9 # for bacteria (typically) / you can change the value
d = 1 # also usual maximum error permission / you can also change this value ;)

def HammingDistance(p,q):
    dist = 0 # start from 0
    for i in range(len(p)): # start matching from the first letter
        if p[i] != q[i]: dist += 1 # if diff. -> add 1 to distance
        else: dist = dist + 0
    return dist

def Neighbors(Pattern, d): # define "neighbors" -> generating a pattern length of 'pl' with 'd' amount of error
    if d == 0: # if there's no 'error'
        return {Pattern}
    if len(Pattern) == 1: # if there's 'error' (difference)
        return {'A', 'C', 'G', 'T'} # possible 'error' options

    neighborhood = set() # remove duplicates
    suffix_neighbors = Neighbors(Pattern[1:], d) # finding neighbors of the suffix (except the first letter)
    for neighbor in suffix_neighbors: # compare found neightbors & origin
        if HammingDistance(Pattern[1:], neighbor) < d: # if the difference is smaller than d
            for base in ['A', 'C', 'G', 'T']: # any base can be used for the first letter
                neighborhood.add(base + neighbor) # so add it!
        else:
            neighborhood.add(Pattern[0] + neighbor) # if the difference is d
    return neighborhood # return it (cannot change the first letter)

def Reverse_Complement(pattern):
    pairs = {"A":"T", "T":"A", "C":"G", "G":"C"} # A-T & G-C
    Complement = "" # new string
    for base in pattern:
        Complement = Complement + pairs[base] # complementary strand sequence generated
    return Complement[::-1] # reversing the order

def FrequentWordsWithMismatchesandRC(text, k, d): # finding the most frequent pattern & also the reversed ones!!
    freq_map = {} # empty dictionary (frequency map/k-mer index)
    for i in range(len(text) - k + 1): # scan through : 1 - (sequence - pl + 1)
        pattern = text[i:i+k] # the part python is currently looking at
        neighborhood = Neighbors(pattern, d) # generate every possible pattern (neighbors)
        for neighbor in neighborhood: # put generated neighbors in the dictionary
            freq_map[neighbor] = freq_map.get(neighbor, 0) + 1 # if there's already in the list -> +1, else 0 -> 1
    combined_freq = {} # new dictionary: original + reversed complementary
    for kmer in freq_map:
        rc_kmer = Reverse_Complement(kmer) # generate reversed complementary (RC) sequences
        combined_freq[kmer] = freq_map.get(kmer, 0) + freq_map.get(rc_kmer, 0) # combine OG + RC frequency
    max_count = max(combined_freq.values()) # find highest frequency
    result = [kmer for kmer, count in combined_freq.items() if count == max_count] # collect highest-frequency-patterns
    return result

final_result = FrequentWordsWithMismatchesandRC(sequence, pl, d) # clean up the result! (unnecessary -> omit it)
print(*final_result) # violaaaaa